# Regime Parallel-Coordinates Plots (L5L6)

Interactive parallel-coordinates views of the regime-stratified sensitivity analysis,
built **entirely from the saved CSVs** in `sa_results/` — no model re-runs. Four views:

- **A** — samples coloured by **regime** (parameter space; the regime manifold)
- **B** — one plot **per regime**, coloured by **log10(flux)** (within-regime structure)
- **C** — **δ sensitivity shift** across regimes (each line = a parameter)
- **D** — **output space** coloured by regime

Interactive: drag on any axis to **brush/filter** the lines. A standalone HTML copy of each
figure is written to `sa_results/figures/`. Run with the **`mace_env`** kernel.

In [1]:
import os, sys, importlib
import numpy as np
import pandas as pd

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import calculations.sensitivity as sens
importlib.reload(sens)
from calculations.sensitivity import (
    parallel_coordinates_samples, parallel_coordinates_sensitivity, top_drivers,
)

In [2]:
RESULTS_DIR = "sa_results"
FIG_DIR     = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

master   = pd.read_csv(os.path.join(RESULTS_DIR, "master_clusters.csv"))
routeB   = pd.read_csv(os.path.join(RESULTS_DIR, "routeB_givendata.csv"))
cmp_flux = pd.read_csv(os.path.join(RESULTS_DIR, "compare_delta_flux.csv"))

# Columns shown on a log10 axis (span orders of magnitude)
LOG_COLS = {'P_upstream', 'k_diss_ref', 'k_diss_metal_ref', 'K_eq_ref', 'K_eq_metal_ref',
            'oxide_thickness', 'metal_thickness', 'D_ref', 'D_ox_ref', 'K_s_ref', 'K_ox_ref',
            'grain_size', 'gb_thickness', 'lattice_density',
            'trap_dislocation_N_T', 'trap_gb_N_T', 'trap_vacancy_N_T', 'trap_carbide_N_T',
            'f_pinhole', 'f_crack', 'f_gb_defect', 'flux', 'permeability', 'D_eff'}
def logsub(dims):
    return [d for d in dims if d in LOG_COLS]

print("master rows:", len(master), "| regimes:", master['regime'].value_counts().to_dict())

master rows: 5252 | regimes: {'oxide': 3725, 'metal': 813, 'surface': 714}


## A — samples coloured by regime

Axes = union of each regime's top-5 δ drivers of flux (+ `log10(flux)`). Each line is one model
run; colour = regime. Brush an axis (e.g. pull `log10(P_upstream)` to its low end) to see which
regime/flux lines survive.

In [3]:
drivers = []
for r in ['surface', 'oxide', 'metal']:
    for p in top_drivers(routeB, r, 'flux', k=5):
        if p not in drivers:
            drivers.append(p)
dimsA = drivers + ['flux']

figA = parallel_coordinates_samples(
    master, dimsA, color_by='regime', log_dims=logsub(dimsA),
    title='A — samples by regime (top δ drivers)',
    save_html=os.path.join(FIG_DIR, 'pcp_A_regime.html'))
print("A axes:", dimsA)
figA

A axes: ['temperature', 'P_upstream', 'f_pinhole', 'k_diss_ref', 'metal_thickness', 'H_sol_ox', 'D_ref', 'trap_dislocation_N_T', 'oxide_thickness', 'trap_gb_N_T', 'flux']


## B — per-regime clusters coloured by log10(flux)

One PCP per regime, axes = that regime's top-6 δ drivers, colour = `log10(flux)`. Clean
within-regime view (no cross-preset mixing).

In [4]:
figsB = {}
for r in ['surface', 'oxide', 'metal']:
    dims = top_drivers(routeB, r, 'flux', k=6)
    sub = master[master['regime'] == r]
    figsB[r] = parallel_coordinates_samples(
        sub, dims, color_by='flux', log_dims=logsub(dims),
        title=f'B — {r} cluster, coloured by log10(flux)',
        save_html=os.path.join(FIG_DIR, f'pcp_B_{r}.html'))
    print(f"{r:8s}: n={len(sub)}  axes={dims}")

for r in ['surface', 'oxide', 'metal']:
    figsB[r].show()

surface : n=714  axes=['temperature', 'P_upstream', 'f_pinhole', 'k_diss_ref', 'metal_thickness', 'trap_dislocation_E_b']
oxide   : n=3725  axes=['temperature', 'H_sol_ox', 'f_pinhole', 'D_ref', 'trap_dislocation_N_T', 'D_ox_ref']
metal   : n=813  axes=['temperature', 'oxide_thickness', 'metal_thickness', 'k_diss_ref', 'trap_gb_N_T', 'f_gb_defect']


## C — δ sensitivity shift across regimes

Each line is a **parameter**; axes are the three regimes; height = its Borgonovo-δ for log10(flux).
Steeply sloped lines are parameters whose importance changes between regimes (e.g. `H_sol_ox` rising
on the oxide axis). Colour = mean importance.

In [5]:
figC = parallel_coordinates_sensitivity(
    cmp_flux, title='C — δ sensitivity shift across regimes (log10 flux)',
    save_html=os.path.join(FIG_DIR, 'pcp_C_sensitivity.html'))
figC

## D — output space coloured by regime

Axes = the model outputs; colour = regime. Shows how the regimes separate in *output* space
(e.g. surface regime → near-zero `theta`).

In [6]:
dimsD = ['flux', 'theta', 'frac_surface', 'frac_oxide', 'frac_metal', 'permeability']
figD = parallel_coordinates_samples(
    master, dimsD, color_by='regime', log_dims=logsub(dimsD),
    title='D — output space by regime',
    save_html=os.path.join(FIG_DIR, 'pcp_D_outputs.html'))
figD

In [7]:
import glob
print("HTML figures written to", os.path.abspath(FIG_DIR), ":")
for f in sorted(glob.glob(os.path.join(FIG_DIR, '*.html'))):
    print("  ", os.path.basename(f), f"({os.path.getsize(f)//1024} KB)")

HTML figures written to /Users/akinyemi.az/Desktop/HE_papers/yoshi/Analytical_model/MHI_permeation/Application/sa_results/figures :
   pcp_A_regime.html (5430 KB)
   pcp_B_metal.html (4790 KB)
   pcp_B_oxide.html (5014 KB)
   pcp_B_surface.html (4784 KB)
   pcp_C_sensitivity.html (4726 KB)
   pcp_D_outputs.html (5169 KB)
